# Graphify Demo (Dataiku) — Knowledge Graph over a Python/PySpark Propensity Pipeline

This version is adapted for Dataiku DSS notebooks, where `!command` subprocess calls
often don't inherit the code env's `PATH` (a known DSS kernel quirk). Every call below
goes through `sys.executable -m graphify ...` instead of the bare `graphify` command,
which sidesteps the issue entirely.

**Before running:** set `REPO_PATH` to the local path where you unzipped your codebase
(see the earlier setup: managed folder → download stream → unzip).

In [ ]:
import sys
print("Kernel Python:", sys.executable)

In [ ]:
REPO_PATH = "./propensity-pipeline"   # <-- confirm/adjust to your unzipped codebase path

import os
print("Resolved path:", os.path.abspath(REPO_PATH))
print("Contents:", os.listdir(REPO_PATH))

## 1. Install / confirm Graphify is available in this kernel

Use `%pip` (kernel-aware magic) for installing — not `!pip` — since `!` spawns a bare
shell that may not share this kernel's environment.

In [ ]:
%pip install graphifyy --quiet

In [ ]:
!{sys.executable} -m graphify --version

## 2. Run Graphify on the codebase

Free and local for pure Python/PySpark code — tree-sitter parsing, no LLM call, no API key.
Only add `--backend claude` (with `ANTHROPIC_API_KEY` set) if the repo also has docs/notebooks
you want semantically linked in.

In [ ]:
!{sys.executable} -m graphify extract {REPO_PATH}

# With semantic extraction for docs/notebooks too:
# import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
# !{sys.executable} -m graphify extract {REPO_PATH} --backend claude

In [ ]:
graphify_out = os.path.join(REPO_PATH, "graphify-out")
print(os.listdir(graphify_out))

## 3. Read the auto-generated report

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

report_path = Path(graphify_out) / "GRAPH_REPORT.md"
display(Markdown(report_path.read_text()))

## 4. Load `graph.json` into NetworkX

In [ ]:
%pip install networkx matplotlib --quiet

In [ ]:
import json
import networkx as nx

graph_path = Path(graphify_out) / "graph.json"
with open(graph_path) as f:
    graph_data = json.load(f)

G = nx.node_link_graph(graph_data)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

In [ ]:
for node_id, attrs in list(G.nodes(data=True))[:5]:
    print(node_id, "->", attrs)

In [ ]:
for u, v, attrs in list(G.edges(data=True))[:5]:
    print(f"{u} --[{attrs.get('relation')} | {attrs.get('confidence')}]--> {v}")

### God nodes (highest-degree)

In [ ]:
degree_ranked = sorted(G.degree, key=lambda x: x[1], reverse=True)[:10]
for node_id, degree in degree_ranked:
    label = G.nodes[node_id].get("label", node_id)
    print(f"{degree:>4}  {label}")

### EXTRACTED vs INFERRED edge breakdown

In [ ]:
from collections import Counter

confidence_counts = Counter(attrs.get("confidence") for _, _, attrs in G.edges(data=True))
print(confidence_counts)

## 5. Query the graph from Python

All subprocess calls go through `sys.executable -m graphify` — not the bare `graphify`
command — for the same PATH reason as above.

In [ ]:
import subprocess

def run_graphify(*args) -> str:
    result = subprocess.run(
        [sys.executable, "-m", "graphify", *args],
        capture_output=True, text=True,
    )
    return result.stdout if result.returncode == 0 else result.stderr

def graphify_query(question: str, graph_json: str = str(graph_path)) -> str:
    return run_graphify("query", question, "--graph", graph_json)

def graphify_path_between(start: str, end: str, graph_json: str = str(graph_path)) -> str:
    return run_graphify("path", start, end, "--graph", graph_json)

def graphify_explain(node_name: str, graph_json: str = str(graph_path)) -> str:
    return run_graphify("explain", node_name, "--graph", graph_json)

In [ ]:
print(graphify_query("what feeds into the propensity score?"))

In [ ]:
# Impact analysis: trace lineage from a raw table to the final model score
print(graphify_path_between("raw_transactions_table", "final_model_score"))

In [ ]:
print(graphify_explain("FeatureEngineeringPipeline"))

## 6. Visualize the graph inline

In [ ]:
import matplotlib.pyplot as plt

top_nodes = [n for n, _ in degree_ranked[:15]]
neighborhood = set(top_nodes)
for n in top_nodes:
    neighborhood.update(G.neighbors(n))

subG = G.subgraph(neighborhood)

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(subG, seed=42, k=0.6)
labels = {n: subG.nodes[n].get("label", n)[:20] for n in subG.nodes}

nx.draw_networkx_nodes(subG, pos, node_size=[subG.degree(n) * 40 for n in subG.nodes], node_color="#4C72B0", alpha=0.85)
nx.draw_networkx_edges(subG, pos, alpha=0.3, arrows=True)
nx.draw_networkx_labels(subG, pos, labels=labels, font_size=7)

plt.title("Propensity Pipeline — God Nodes + Neighborhood")
plt.axis("off")
plt.tight_layout()
plt.show()

### Or embed the full interactive `graph.html`

In [ ]:
from IPython.display import IFrame

IFrame(src=str(Path(graphify_out) / "graph.html"), width=1000, height=700)

## 7. Community detection

Requires the `leiden` extra (Python < 3.13 only): `%pip install "graphifyy[leiden]"`

In [ ]:
%pip install "graphifyy[leiden]" --quiet

In [ ]:
!{sys.executable} -m graphify {REPO_PATH} --cluster-only --resolution 1.5

In [ ]:
with open(graph_path) as f:
    graph_data = json.load(f)
G = nx.node_link_graph(graph_data)

communities = Counter(attrs.get("community_name", "unlabeled") for _, attrs in G.nodes(data=True))
for name, count in communities.most_common():
    print(f"{count:>4}  {name}")

## 8. Architecture doc (Mermaid call-flow HTML)

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "graphify", "export", "callflow-html"],
    cwd=REPO_PATH,
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)

## Summary — what to show in the demo

| Step | What it proves |
|---|---|
| `graphify extract` | Local, free parsing of the whole pipeline into a graph |
| `GRAPH_REPORT.md` | Instant orientation: god nodes, surprising connections, why-comments |
| `graphify_query(...)` | Ask architecture questions in plain English |
| `graphify_path_between(...)` | Feature lineage / impact analysis — "what does changing this break" |
| EXTRACTED vs INFERRED | Audit trail — ground truth vs. reasoned guess, with confidence scores |
| Community detection | Auto-discovered subsystems, no embeddings needed |
| `callflow-html` export | Visual artifact for stakeholders |

**Dataiku-specific gotchas hit along the way (worth remembering):**
- `%pip install` (kernel-aware) instead of `!pip install` (bare shell, may miss the env)
- `!{sys.executable} -m graphify ...` instead of bare `!graphify ...` (PATH not inherited by `!`)
- Managed folders backed by HDFS need `get_download_stream()` — `get_path()` fails on them